In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\DDoS\\combined_ddos.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "ddos_lstm.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\DDoS\combined_ddos.csv | Total rows: 100000
Train: 60000 | Val: 20000 | Test: 20000


In [3]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_scaled).unsqueeze(1)  # Add channel dimension for CNN
y_train_t = torch.FloatTensor(y_train.values.copy())
X_val_t   = torch.FloatTensor(X_val_scaled).unsqueeze(1)
y_val_t   = torch.FloatTensor(y_val.values.copy())
X_test_t  = torch.FloatTensor(X_test_scaled).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

In [4]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=128,
                    num_layers=2, batch_first=True, dropout=0.4)
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # take last timestep
        return self.fc(out).squeeze(1)

model     = LSTMModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

In [5]:
epochs = 50
patience = 5
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t.to(device))
        val_loss = criterion(val_preds, y_val_t.to(device)).item()
        val_acc   = ((val_preds > 0.5) == y_val_t.to(device)).float().mean().item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# Load best model before evaluation
model.load_state_dict(torch.load(MODEL_PATH))

Epoch 1/50 | Loss: 0.5527 | Val Loss: 0.5088 | Val Acc: 0.7303
Epoch 2/50 | Loss: 0.4977 | Val Loss: 0.4856 | Val Acc: 0.7395
Epoch 3/50 | Loss: 0.4839 | Val Loss: 0.4749 | Val Acc: 0.7476
Epoch 4/50 | Loss: 0.4743 | Val Loss: 0.4689 | Val Acc: 0.7538
Epoch 5/50 | Loss: 0.4667 | Val Loss: 0.4629 | Val Acc: 0.7592
Epoch 6/50 | Loss: 0.4623 | Val Loss: 0.4584 | Val Acc: 0.7620
Epoch 7/50 | Loss: 0.4577 | Val Loss: 0.4532 | Val Acc: 0.7628
Epoch 8/50 | Loss: 0.4519 | Val Loss: 0.4506 | Val Acc: 0.7769
Epoch 9/50 | Loss: 0.4470 | Val Loss: 0.4453 | Val Acc: 0.7753
Epoch 10/50 | Loss: 0.4422 | Val Loss: 0.4412 | Val Acc: 0.7829
Epoch 11/50 | Loss: 0.4363 | Val Loss: 0.4366 | Val Acc: 0.7842
Epoch 12/50 | Loss: 0.4330 | Val Loss: 0.4395 | Val Acc: 0.7807
Epoch 13/50 | Loss: 0.4297 | Val Loss: 0.4306 | Val Acc: 0.7874
Epoch 14/50 | Loss: 0.4285 | Val Loss: 0.4310 | Val Acc: 0.7835
Epoch 15/50 | Loss: 0.4249 | Val Loss: 0.4282 | Val Acc: 0.7894
Epoch 16/50 | Loss: 0.4222 | Val Loss: 0.4242 | V

<All keys matched successfully>

In [6]:
model.eval()
with torch.no_grad():
    preds = (model(X_test_t.to(device)) > 0.5).cpu().numpy().astype(int)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.74      0.94      0.83     10000
      Attack       0.91      0.67      0.77     10000

    accuracy                           0.80     20000
   macro avg       0.83      0.80      0.80     20000
weighted avg       0.83      0.80      0.80     20000

[[9365  635]
 [3298 6702]]
